In [1]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER
import cftime
import datetime
from datetime import date
from matplotlib import pyplot
import numpy
import pandas
import xarray as xr

In [2]:
Diri = 'Storm_Data_Raw/'
Output_Diri = 'Storm_Tracking/'

In [3]:
# Open file
def Create_DF(File):
    Data = open(File, 'r')
    Rows = []
    Storm_Code, Storm_List_Orig = [], []
#
# Get storm codes
    for Line in Data:
        Line = Line.strip()
        if Line.startswith("start"):
            Code = Line.split()[-1]
            Storm_List_Orig.append(Code)
        else:
            Storm_Code.append(Code)
            Values = Line.split()
            Rows.append(Values)
#
# Create dataframe
    Cols = ["Lon", "Lat", "SLP(hPa)", "Winds(m/s)", "Dist(m)", "Angle", "B", "VLT", "VUT", "Year", "Month", "Day", "Hour"]
    DF = pandas.DataFrame(Rows, columns=Cols)
    DF = DF.apply(pandas.to_numeric)
    DF["Time(Z)"] = pandas.to_datetime(DF[["Year", "Month", "Day", "Hour"]])
    DF = DF.drop(columns=["Year", "Month", "Day", "Hour"])
    DF.insert(0, "Code", Storm_Code)
#
# Remove data points missing phase space parameter values (-999.0) or SLP > 1013hPa
    Mask = (DF != -999.00).all(axis=1)
    DF = DF[Mask].reset_index(drop=True)
    DF = DF[DF["SLP(hPa)"] <= 1013].reset_index(drop=True)
    return (DF, Storm_List_Orig)

In [4]:
# Combine the three datasets into one dataframe of 90 or 93 years for each scenario
def Combine_DF(DF_001, DF_002, DF_003, Orig_List_001, Orig_List_002, Orig_List_003, Year_Start):
    Orig_Combine, New_Combine, Ensemble = [], [], []
    DF_Combine = DF_001.iloc[0:0].copy()
    DF_Combine.insert(1, "Ensemble", [])
#
# Ensemble 001
    Year = Year_Start
    Orig_Year = DF_001["Time(Z)"][0].year
    Year_Diff = Year - Orig_Year
    Count = 0
    for k in range(len(Orig_List_001)):
        Orig_Code = Orig_List_001[k]
        DF_Storm = DF_001[DF_001["Code"] == Orig_Code].reset_index(drop=True)
        if len(DF_Storm) > 0:
            New_Code, Year, Count = Update_Code(DF_Storm, Year_Diff, Year, Count)
            Orig_Combine.append(Orig_Code)
            New_Combine.append(New_Code)
            Ensemble.append(1)
            DF_Combine = Concat_DF(DF_Combine, DF_Storm, New_Code, Year_Diff, 1)
#
# Ensemble 002
    Year = Year_Start + 32
    Orig_Year = DF_002["Time(Z)"][0].year
    Year_Diff = Year - Orig_Year
    Count = 0
    for k in range(len(Orig_List_002)):
        Orig_Code = Orig_List_002[k]
        DF_Storm = DF_002[DF_002["Code"] == Orig_Code].reset_index(drop=True)
        if len(DF_Storm) > 0:
            New_Code, Year, Count = Update_Code(DF_Storm, Year_Diff, Year, Count)
            Orig_Combine.append(Orig_Code)
            New_Combine.append(New_Code)
            Ensemble.append(2)
            DF_Combine = Concat_DF(DF_Combine, DF_Storm, New_Code, Year_Diff, 2)
#
# Ensemble 003
    Year = Year_Start + 64
    Orig_Year = DF_003["Time(Z)"][0].year
    Year_Diff = Year - Orig_Year
    Count = 0
    for k in range(len(Orig_List_003)):
        Orig_Code = Orig_List_003[k]
        DF_Storm = DF_003[DF_003["Code"] == Orig_Code].reset_index(drop=True)
        if len(DF_Storm) > 0:
            New_Code, Year, Count = Update_Code(DF_Storm, Year_Diff, Year, Count)
            Orig_Combine.append(Orig_Code)
            New_Combine.append(New_Code)
            Ensemble.append(3)
            DF_Combine = Concat_DF(DF_Combine, DF_Storm, New_Code, Year_Diff, 3)

# Create dataframe to store data
    Codes_DF = pandas.DataFrame({"Ensemble": Ensemble, "Orig Code": Orig_Combine, "New Code": New_Combine})
    return (DF_Combine, Codes_DF)

In [5]:
# Update storm code to contain year and storm number
def Update_Code(DF_Storm, Year_Diff, Year, Count):
    Orig_Year = DF_Storm["Time(Z)"][0].year
    Time_Year = Orig_Year + Year_Diff
    if Time_Year == Year:
        Count += 1
    else:
        Year += 1
        Count = 1
    Code = Code = f"TC{Year}{Count:02d}"
    return (Code, Year, Count)

In [6]:
# Concatenate dataframes
def Concat_DF(DF_Combine, DF_Storm, New_Code, Year_Diff, Ensemble):
    New_Time, New_Codes, Ensembles = [], [], []
    for i in range(len(DF_Storm)):
        New_Time.append(Update_Year(DF_Storm["Time(Z)"][i], Year_Diff))
        New_Codes.append(New_Code)
        Ensembles.append(Ensemble)
    DF_Storm["Code"] = New_Codes
    DF_Storm["Time(Z)"] = New_Time
    DF_Storm["Ensemble"] = Ensembles
    DF_Combine = pandas.concat([DF_Combine, DF_Storm], ignore_index=True)
    return (DF_Combine)

In [7]:
# Change year of data
def Update_Year(Orig_Time, Year_Diff):
    Year_Update = Orig_Time.year + Year_Diff
    New_Time = Orig_Time.replace(year=Year_Update)
    return (New_Time)

In [8]:
# Open name list file
def Open_Name_List(File):
    Name_File = pandas.read_csv(File)
    Headings = list(Name_File.columns)
    Name_List = []
    for i in range(len(Headings)):
        for j in range(len(Name_File)):
            Name_List.append(Name_File[Headings[i]][j])
    return (Name_List)

In [9]:
# Find distance between two points
def Find_Distance(y1, y2, x1, x2):
    Start_Lat = y1 * numpy.pi / 180
    End_Lat = y2 * numpy.pi / 180
    Start_Lon = x1 * numpy.pi / 180
    End_Lon = x2 * numpy.pi / 180
    Lat_Diff = End_Lat - Start_Lat
    Lon_Diff = End_Lon - Start_Lon
    Earth_Rad = 6378
    Distance = 2 * Earth_Rad * numpy.sqrt((numpy.sin(Lat_Diff/2))**2 + \
    numpy.cos(Start_Lat) * numpy.cos(End_Lat) * (numpy.sin(Lon_Diff/2))**2)
    return (Distance)

In [10]:
# Find a specific storm within the dataframe
def Find_Storm(DF, Code):
    DF_Storm = DF[DF["Code"] == Code].reset_index(drop=True)
    return (DF_Storm)

In [11]:
# Open name list
Name_List = Open_Name_List(Diri + 'Storm_Name_List.csv')

In [12]:
# Open control datasets
Control_001_Data_V0, Control_001_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.control_avg')
Control_002_Data_V0, Control_002_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.control.002_avg')
Control_003_Data_V0, Control_003_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.control.004_avg')
Control_Data_V1, Control_Codes_V1 = \
Combine_DF(Control_001_Data_V0, Control_002_Data_V0, Control_003_Data_V0, \
Control_001_Storm_List_V0, Control_002_Storm_List_V0, Control_003_Storm_List_V0, 1901)

In [13]:
# Open RCP4.5 datasets
RCP45_001_Data_V0, RCP45_001_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.rcp45_avg')
RCP45_002_Data_V0, RCP45_002_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.rcp45.002_avg')
RCP45_003_Data_V0, RCP45_003_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.rcp45.003_avg')
RCP45_Data_V1, RCP45_Codes_V1 = \
Combine_DF(RCP45_001_Data_V0, RCP45_002_Data_V0, RCP45_003_Data_V0, \
RCP45_001_Storm_List_V0, RCP45_002_Storm_List_V0, RCP45_003_Storm_List_V0, 2002)

In [14]:
# Open RCP8.5 datasets
RCP85_001_Data_V0, RCP85_001_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.rcp85_avg')
RCP85_002_Data_V0, RCP85_002_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.rcp85.003_avg')
RCP85_003_Data_V0, RCP85_003_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.rcp85.004_avg')
RCP85_Data_V1, RCP85_Codes_V1 = \
Combine_DF(RCP85_001_Data_V0, RCP85_002_Data_V0, RCP85_003_Data_V0, \
RCP85_001_Storm_List_V0, RCP85_002_Storm_List_V0, RCP85_003_Storm_List_V0, 2102)

In [15]:
# Create DF with extratropical transition information of each storm
def Storm_Info_DF(DF, Codes):
    Ensemble = []
    Trans_Types, Trop_Begin_Times, ET_Begin_Times, ET_Compl_Times, Track_Discons = [], [], [], [], []
    for i in range(len(Codes)):
        DF_Storm_Orig = DF[DF["Code"] == Codes["New Code"][i]].reset_index(drop=True)
        DF_Storm, Track_Discons = DF_Fixing(DF_Storm_Orig, Track_Discons)
        Trans_Type, Trop_Begin_t, ET_Begin_t, ET_Compl_t = ET_Function(DF_Storm)
        if Trop_Begin_t >= 0:
            Trop_Begin_Time = DF_Storm["Time(Z)"][Trop_Begin_t]
        if ET_Begin_t > 0:
            ET_Begin_Time = DF_Storm["Time(Z)"][ET_Begin_t]
        else:
            ET_Begin_Time = numpy.nan
        if ET_Compl_t > 0:
            ET_Compl_Time = DF_Storm["Time(Z)"][ET_Compl_t]
        else:
            ET_Compl_Time = numpy.nan
        Trop_Begin_Times.append(Trop_Begin_Time)
        ET_Begin_Times.append(ET_Begin_Time)
        ET_Compl_Times.append(ET_Compl_Time)
        Trans_Types.append(Trans_Type)
        Ensemble.append(Codes["Ensemble"][i])
        if i == 0:
            DF_Fixed = DF_Storm.copy()
        else:
            DF_Fixed = pandas.concat([DF_Fixed, DF_Storm])
    DF_Fixed = DF_Fixed.reset_index(drop=True)
#
# Create Storms DF
    Storms_DF_Orig = pandas.DataFrame({"Code": Codes["New Code"], "Orig Code": Codes["Orig Code"], "Ensemble": Ensemble, \
    "Trans Type": Trans_Types, "Trop Begin Time": Trop_Begin_Times, "ET Begin Time": ET_Begin_Times, "ET Complete Time": ET_Compl_Times})
    return (DF_Fixed, Storms_DF_Orig)

In [16]:
# Check for time discontinuity
def Time_Discontinuity(DF_Storm):
    Time_Continuous = 0
    for k in range(len(DF_Storm)):
        if k > 0 and Time_Continuous == 0:
# If over 12 hours of missing data
            if DF_Storm["Time(Z)"][k] - DF_Storm["Time(Z)"][k-1] >= datetime.timedelta(hours=12):
# Divide array into before the time discontinuity and after it
                Before = DF_Storm["Time(Z)"][k-1]
                After = DF_Storm["Time(Z)"][k]
                DF_Before = DF_Storm[DF_Storm["Time(Z)"] <= Before].reset_index(drop=True)
                DF_After = DF_Storm[DF_Storm["Time(Z)"] >= After].reset_index(drop=True)
# Choose the part with the lowest min SLP to keep
                if numpy.min(DF_After["SLP(hPa)"]) <= numpy.min(DF_Before["SLP(hPa)"]) and \
                len(DF_After) > len(DF_Before):
                    DF_Storm_Updated = DF_After
                else:
                    DF_Storm_Updated = DF_Before
                Time_Continuous = 1
# Repeat until no more time discontinuity
    if Time_Continuous == 0:
        DF_Storm_Updated = DF_Storm.copy()
    return (DF_Storm_Updated)

In [17]:
# Check for track discontinuity
def Track_Discontinuity(DF_Storm):
    Track_Continuous = 0
    Track_Discon = []
    for k in range(len(DF_Storm)):
        if k > 1 and Track_Continuous == 0:
# Calculate distance between each 6 hour interval
            Distance_1 = Find_Distance(DF_Storm["Lat"][k], DF_Storm["Lat"][k-1], \
            DF_Storm["Lon"][k], DF_Storm["Lon"][k-1])
            Distance_2 = Find_Distance(DF_Storm["Lat"][k-1], DF_Storm["Lat"][k-2], \
            DF_Storm["Lon"][k-1], DF_Storm["Lon"][k-2])
# If distance >= 800km and distance over 5 times further than previous time interval
            if Distance_1 >= 800 and Distance_1 >= Distance_2 * 5:
                Track_Discon = [DF_Storm["Code"][0], DF_Storm["Time(Z)"][k]]
# Divide array into before the track discontinuity and after it
                Before = DF_Storm["Time(Z)"][k-1]
                After = DF_Storm["Time(Z)"][k]
                DF_Before = DF_Storm[DF_Storm["Time(Z)"] <= Before].reset_index(drop=True)
                DF_After = DF_Storm[DF_Storm["Time(Z)"] >= After].reset_index(drop=True)
# Choose the part with the lowest min SLP to keep
                if numpy.min(DF_After["SLP(hPa)"]) <= numpy.min(DF_Before["SLP(hPa)"]) and \
                len(DF_After) > len(DF_Before):
                    DF_Storm_Updated = DF_After
                else:
                    DF_Storm_Updated = DF_Before
                Track_Continuous = 1
# Repeat until no more track discontinuity
    if Track_Continuous == 0:
        DF_Storm_Updated = DF_Storm.copy()
    return (DF_Storm_Updated, Track_Discon)

In [18]:
# Fix storms that have time discontinuity or track discontinuity
def DF_Fixing(DF_Storm_Orig, Track_Discons):
    Bottom_Right = False
    Top_Right = False
    Bottom_Left = False
    Top_Left = False
    ET_Complete = False
#
# Check for time discontinuity
    DF_Storm = Time_Discontinuity(DF_Storm_Orig)
# Repeat to check again
    DF_Storm = Time_Discontinuity(DF_Storm)
    DF_Storm = Time_Discontinuity(DF_Storm)
#
# Check for track discontinuity
    DF_Storm, Track_Discon = Track_Discontinuity(DF_Storm)
    if len(Track_Discon) > 0:
        Track_Discons.append(Track_Discon)
# Repeat to check again
    DF_Storm, Track_Discon = Track_Discontinuity(DF_Storm)
    if len(Track_Discon) > 0:
        Track_Discons.append(Track_Discon)
    DF_Storm, Track_Discon = Track_Discontinuity(DF_Storm)
    if len(Track_Discon) > 0:
        Track_Discons.append(Track_Discon)
    return (DF_Storm, Track_Discons)

In [19]:
# Function to find whether a storm has completed ET, and start and end time of ET
def ET_Function(DF_Storm):
    Tropical_Begin = False
    ET_Begin = False
    ET_Complete = False
    Bottom_Right_t, Top_Right_t, Bottom_Left_t = [], [], []
    Trop_Begin_t, ET_Begin_t, ET_Compl_t, Trans_Type = -728, -728, -728, -728
    for t in range(len(DF_Storm)):
        if ET_Complete == False:
            if Tropical_Begin == False:
# Storm must first be tropical
                if DF_Storm["B"][t] <= 15 and DF_Storm["VLT"][t] >= 0:
                    Tropical_Begin = True
                    Trop_Begin_t = t
                    Bottom_Right_t.append(t)
            else:
# Storm must be tropical for at least 48 hours
                if len(Bottom_Right_t) < 8:
                    if DF_Storm["B"][t] <= 15 and DF_Storm["VLT"][t] >= 0:
                        Bottom_Right_t.append(t)
#
# Enter next stage if storm remains tropical for at least 48 hours
                else:
                    if ET_Begin == False:
# If storm enters top right quadrant = ET Begin
                        if DF_Storm["B"][t] > 15 and DF_Storm["VLT"][t] >= 0:
                            ET_Begin = True
                            ET_Begin_t = t
                            Top_Right_t.append(t)
# If storm enters bottom left quadrant = ET Begin
                        elif DF_Storm["B"][t] <= 15 and DF_Storm["VLT"][t] < 0:
                            ET_Begin = True
                            ET_Begin_t = t
                            Bottom_Left_t.append(t)
# If storm directly enters top left quadrant = ET Complete
                        elif DF_Storm["B"][t] > 15 and DF_Storm["VLT"][t] < 0:
                            ET_Complete = True
                            ET_Begin_t, ET_Compl_t = t, t
#
# After ET initiation
                    elif ET_Begin == True:
                        if len(Top_Right_t) > 0 and len(Top_Right_t) >= len(Bottom_Left_t):
# If storm remains in top right quadrant
                            if DF_Storm["B"][t] > 15 and DF_Storm["VLT"][t] >= 0:
                                Top_Right_t.append(t)
# If storm enters top left quadrant = ET Complete
                            elif DF_Storm["VLT"][t] < 0:
                                if DF_Storm["B"][t] > 10:
                                    ET_Complete = True
                                    ET_Compl_t = t
# If storm enters bottom left quadrant
                                else:
                                    Bottom_Left_t.append(t)
                            elif DF_Storm["B"][t] <= 15 and DF_Storm["VLT"][t] >= 0:
# If storm returns to bottom right quadrant for over 24 hours
                                if t > Top_Right_t[-1] + 4:
                                    ET_Begin = False
                                    ET_Begin_t = -728
                                    Top_Right_t = []
                        elif len(Bottom_Left_t) > 0:
# If storm remains in bottom left quadrant
                            if DF_Storm["B"][t] <= 15 and DF_Storm["VLT"][t] < 0:
                                Bottom_Left_t.append(t)
# If storm enters top left quadrant = ET Complete
                            elif DF_Storm["B"][t] > 15:
                                if DF_Storm["VLT"][t] <= 0:
                                    ET_Complete = True
                                    ET_Compl_t = t
# If storm enters top right quadrant
                                else:
                                    Top_Right_t.append(t)
                            elif DF_Storm["B"][t] <= 15 and DF_Storm["VLT"][t] >= 0:
# If storm returns to bottom right quadrant for over 24 hours
                                if t-4 > Bottom_Left_t[len(Bottom_Left_t)-1]:
                                    ET_Begin = False
                                    ET_Begin_t = -728
                                    Bottom_Left_t = []
#
# If storm did not spend at least 48 hours as tropical
    if len(Bottom_Right_t) < 8:
        Trans_Type = -728
# If ET never began
    elif ET_Compl_t == -728:
        if ET_Begin_t == -728:
            Trans_Type = 0
# If ET began but did not complete
        else:
            Trans_Type = 1
    else:
        Trans_Type = 2
    return (Trans_Type, Trop_Begin_t, ET_Begin_t, ET_Compl_t)

In [20]:
# Create DF with extratropical transition information of each storm
Control_Data_V2, Control_ET_V1 = Storm_Info_DF(Control_Data_V1, Control_Codes_V1)
RCP45_Data_V2, RCP45_ET_V1 = Storm_Info_DF(RCP45_Data_V1, RCP45_Codes_V1)
RCP85_Data_V2, RCP85_ET_V1 = Storm_Info_DF(RCP85_Data_V1, RCP85_Codes_V1)

In [21]:
# Add storm phase into dataframe
def Storm_Phase_Info(Data_DF_Orig, ET_DF):
    Data_DF = Data_DF_Orig.copy()
    Storm_Phase_List = []
    for i in range(len(ET_DF)):
        DF_Storm = Data_DF[Data_DF["Code"] == ET_DF["Code"][i]].reset_index(drop=True)
# Find storm phase based on ET start time and ET end time
        Storm_Phase = Find_Storm_Phase(DF_Storm, ET_DF["Trop Begin Time"][i], ET_DF["ET Begin Time"][i], ET_DF["ET Complete Time"][i], ET_DF["Trans Type"][i])
        for j in range(len(Storm_Phase)):
            Storm_Phase_List.append(Storm_Phase[j])
    Data_DF["Storm Phase"] = Storm_Phase_List
    return (Data_DF)

In [22]:
# Function to find storm phase
def Find_Storm_Phase(DF_Storm, Trop_Begin_Time, ET_Begin_Time, ET_Compl_Time, Trans_Type):
    Time = list(DF_Storm["Time(Z)"])
    Storm_Phase = []
    for k in range(len(DF_Storm)):
# If before first tropical = pre-tropical
# If before ET begin = tropical
# If between ET begin and ET complete = transitioning
# If after ET complete = extratropical
        if Time[k] < Trop_Begin_Time:
            Storm_Phase.append("Pre-Tropical")
        else:
            if Trans_Type == 0:
                Storm_Phase.append("Tropical")
            elif Trans_Type == 1:
                if Time[k] < ET_Begin_Time:
                    Storm_Phase.append("Tropical")
                else:
                    Storm_Phase.append("Transition")
            else:
                if Time[k] < ET_Begin_Time:
                    Storm_Phase.append("Tropical")
                elif Time[k] < ET_Compl_Time:
                    Storm_Phase.append("Transition")
                else:
                    Storm_Phase.append("Extratropical")
    return (Storm_Phase)

In [23]:
# Add storm phase into dataframe
Control_Data_V3 = Storm_Phase_Info(Control_Data_V2, Control_ET_V1)
RCP45_Data_V3 = Storm_Phase_Info(RCP45_Data_V2, RCP45_ET_V1)
RCP85_Data_V3 = Storm_Phase_Info(RCP85_Data_V2, RCP85_ET_V1)

In [24]:
# Add lon, lat, SLP into ET dataframe
def ET_Variables(Main_DF, ET_DF_Orig):
    ET_DF = ET_DF_Orig.copy()
    Array = numpy.zeros((21, len(ET_DF)))
    Trop_Peak_Times = []
    Peak_Times = []
    Genesis_Times = []
    Final_Times = []
    for i in range(len(ET_DF)):
        Code = ET_DF["Code"][i]
        Trans_Type, ET_Begin_Time, ET_Compl_Time, Trop_Begin_Time = \
        ET_DF["Trans Type"][i], ET_DF["ET Begin Time"][i], ET_DF["ET Complete Time"][i], ET_DF["Trop Begin Time"][i]
        Storm = Main_DF[Main_DF["Code"] == Code].reset_index(drop=True)
#
# Find lon, lat, time at storm peak
        Min_SLP = numpy.min(Storm["SLP(hPa)"])
        Storm_Peak = Storm[Storm["SLP(hPa)"] == Min_SLP]
        Array[0][i] = Min_SLP
        Array[1][i] = float(Storm_Peak["Lon"].iloc[0])
        Array[2][i] = float(Storm_Peak["Lat"].iloc[0])
        Peak_Times.append(Storm_Peak["Time(Z)"].iloc[0])
#
# Find lon, lat, time at tropical phase peak
        Storm_Trop = Storm[Storm["Storm Phase"] == "Tropical"]
        Trop_Min_SLP = numpy.min(Storm_Trop["SLP(hPa)"])
        Trop_Peak = Storm[Storm["SLP(hPa)"] == Trop_Min_SLP]
        if len(Storm_Trop) > 0:
            Array[3][i] = Trop_Min_SLP
            Array[4][i] = float(Trop_Peak["Lon"].iloc[0])
            Array[5][i] = float(Trop_Peak["Lat"].iloc[0])
            Trop_Peak_Times.append(Trop_Peak["Time(Z)"].iloc[0])
        else:
            Array[3][i] = numpy.nan
            Array[4][i] = numpy.nan
            Array[5][i] = numpy.nan
            Trop_Peak_Times.append(numpy.nan)
#
# Find SLP, lon, lat at genesis
        Array[6][i] = float(Storm["SLP(hPa)"].iloc[0])
        Array[7][i] = float(Storm["Lon"].iloc[0])
        Array[8][i] = float(Storm["Lat"].iloc[0])
        Genesis_Times.append(Storm["Time(Z)"].iloc[0])
#
# Find SLP, lon, lat at final
        Array[9][i] = float(Storm["SLP(hPa)"].iloc[-1])
        Array[10][i] = float(Storm["Lon"].iloc[-1])
        Array[11][i] = float(Storm["Lat"].iloc[-1])
        Final_Times.append(Storm["Time(Z)"].iloc[-1])
#
#
# Find SLP, lon, lat at tropical begin
        Trop_Begin = Storm[Storm["Time(Z)"] == Trop_Begin_Time]
        if len(Trop_Begin) > 0:
            Array[12][i] = float(Trop_Begin["SLP(hPa)"].iloc[0])
            Array[13][i] = float(Trop_Begin["Lon"].iloc[0])
            Array[14][i] = float(Trop_Begin["Lat"].iloc[0])
        else:
            Array[12][i], Array[13][i], Array[14][i] = numpy.nan, numpy.nan, numpy.nan
#
# Find SLP, lon, lat at ET begin
        if Trans_Type >= 1:
            Begin = Storm[Storm["Time(Z)"] == ET_Begin_Time]
            Array[15][i] = float(Begin["SLP(hPa)"].iloc[0])
            Array[16][i] = float(Begin["Lon"].iloc[0])
            Array[17][i] = float(Begin["Lat"].iloc[0])
        else:
            Array[15][i], Array[16][i], Array[17][i] = numpy.nan, numpy.nan, numpy.nan
#
# Find SLP, lon, lat at ET complete
        if Trans_Type == 2:
            Complete = Storm[Storm["Time(Z)"] == ET_Compl_Time]
            Array[18][i] = float(Complete["SLP(hPa)"].iloc[0])
            Array[19][i] = float(Complete["Lon"].iloc[0])
            Array[20][i] = float(Complete["Lat"].iloc[0])
        else:
            Array[18][i], Array[19][i], Array[20][i] = numpy.nan, numpy.nan, numpy.nan
#
# Append into ET DF
    ET_DF["Peak Time"] = Peak_Times
    ET_DF["Trop Peak Time"] = Trop_Peak_Times
    ET_DF["Genesis Time"] = Genesis_Times
    ET_DF["Final Time"] = Final_Times
    Cols = ["Peak SLP", "Peak Lon", "Peak Lat", "Trop Peak SLP", "Trop Peak Lon", "Trop Peak Lat", \
    "Genesis SLP", "Genesis Lon", "Genesis Lat", "Final SLP", "Final Lon", "Final Lat", "Trop Begin SLP", "Trop Begin Lon", "Trop Begin Lat", \
    "ET Begin SLP", "ET Begin Lon", "ET Begin Lat", "ET Complete SLP", "ET Complete Lon", "ET Complete Lat"]
    ET_DF_Cols = pandas.DataFrame(Array.T, columns=Cols, index=ET_DF.index)
    ET_DF = pandas.concat([ET_DF, ET_DF_Cols], axis=1)
# Rearrange columns
    ET_DF = Move_Col(ET_DF, "Peak Time", "Peak SLP")
    ET_DF = Move_Col(ET_DF, "Trop Peak Time", "Trop Peak SLP")
    ET_DF = Move_Col(ET_DF, "Genesis Time", "Genesis SLP")
    ET_DF = Move_Col(ET_DF, "Final Time", "Final SLP")
    ET_DF = Move_Col(ET_DF, "Trop Begin Time", "Trop Peak SLP")
    ET_DF = Move_Col(ET_DF, "ET Begin Time", "ET Begin SLP")
    ET_DF = Move_Col(ET_DF, "ET Complete Time", "ET Complete SLP")
    return (ET_DF)

In [25]:
# Rearrange columns
def Move_Col(DF, Move_Col, Ref_Col):
    Col = DF.pop(Move_Col)
    Index = DF.columns.get_loc(Ref_Col)
    DF.insert(Index, Move_Col, Col)
    return (DF)

In [26]:
# Add lon, lat, SLP into ET dataframe
Control_ET_V2 = ET_Variables(Control_Data_V3, Control_ET_V1)
RCP45_ET_V2 = ET_Variables(RCP45_Data_V3, RCP45_ET_V1)
RCP85_ET_V2 = ET_Variables(RCP85_Data_V3, RCP85_ET_V1)

In [27]:
# Filter out storms that have not been tropical for at least 48 hours
def Tropical_Filter(Data_DF_Orig, ET_DF_Orig, Codes_DF_Orig):
    Filtered_ET_0 = ET_DF_Orig[ET_DF_Orig["Trans Type"] > -728].reset_index(drop=True)
# Filter out storms with tropical peak SLP >= 1000hPa
    Filtered_ET = Filtered_ET_0[Filtered_ET_0["Trop Peak SLP"] < 1000].reset_index(drop=True)
    Code_List = Filtered_ET["Code"]
    Filtered_DF = Data_DF_Orig[Data_DF_Orig["Code"].isin(Code_List)].reset_index(drop=True)
    Filtered_Codes = Codes_DF_Orig[Codes_DF_Orig["New Code"].isin(Code_List)].reset_index(drop=True)
    return (Filtered_DF, Filtered_ET, Filtered_Codes)

In [28]:
# Filter out storms that have not been tropical for at least 48 hours
Control_Data_V4, Control_ET_V4, Control_Codes_V4 = Tropical_Filter(Control_Data_V3, Control_ET_V2, Control_Codes_V1)
RCP45_Data_V4, RCP45_ET_V4, RCP45_Codes_V4 = Tropical_Filter(RCP45_Data_V3, RCP45_ET_V2, RCP45_Codes_V1)
RCP85_Data_V4, RCP85_ET_V4, RCP85_Codes_V4 = Tropical_Filter(RCP85_Data_V3, RCP85_ET_V2, RCP85_Codes_V1)

In [29]:
# Assign names to each storm code
def Assign_Name(Name_List, Data_DF, ET_DF, Codes_DF):
    Storm_Names = []
    for i in range(len(Codes_DF)):
        Name = Name_List[int(i%len(Name_List))]
        Storm_Names.append(Name)
    Codes_DF["Name"] = Storm_Names
    ET_DF.insert(1, "Name", Storm_Names)
    DF_Names = []
    for k in range(len(Data_DF)):
        Name = Codes_DF[Codes_DF["New Code"] == Data_DF["Code"][k]]["Name"].iloc[0]
        DF_Names.append(Name)
    Data_DF.insert(1, "Name", DF_Names)
    return (Data_DF, ET_DF, Codes_DF)

In [30]:
# Assign names to each storm code
Control_Data_V5, Control_ET_V5, Control_Codes_V5 = Assign_Name(Name_List, Control_Data_V4, Control_ET_V4, Control_Codes_V4)
RCP45_Data_V5, RCP45_ET_V5, RCP45_Codes_V5 = Assign_Name(Name_List, RCP45_Data_V4, RCP45_ET_V4, RCP45_Codes_V4)
RCP85_Data_V5, RCP85_ET_V5, RCP85_Codes_V5 = Assign_Name(Name_List, RCP85_Data_V4, RCP85_ET_V4, RCP85_Codes_V4)

In [31]:
# Filter for storms that completed ET
def ET_Filter(Data_DF_Orig, ET_DF_Orig):
    ET_DF = ET_DF_Orig[ET_DF_Orig["Trans Type"] == 2].reset_index(drop=True)
    Code_List = ET_DF["Code"]
    Filtered_DF = Data_DF_Orig[Data_DF_Orig["Code"].isin(Code_List)].reset_index(drop=True)
    return (Filtered_DF, ET_DF)

In [32]:
# Filter for storms that completed ET
Control_Data_V6, Control_ET_V6 = ET_Filter(Control_Data_V5, Control_ET_V5)
RCP45_Data_V6, RCP45_ET_V6 = ET_Filter(RCP45_Data_V5, RCP45_ET_V5)
RCP85_Data_V6, RCP85_ET_V6 = ET_Filter(RCP85_Data_V5, RCP85_ET_V5)

In [33]:
# Filter for storms with ET Begin SLP <= 1005 and ET Complete SLP <= 1005
def SLP_Filter(Data_DF_Orig, ET_DF_Orig):
    ET_DF = ET_DF_Orig[(ET_DF_Orig["ET Begin SLP"] <= 1005) & (ET_DF_Orig["ET Complete SLP"] <= 1005)].reset_index(drop=True)
    Code_List = ET_DF["Code"]
    Filtered_DF = Data_DF_Orig[Data_DF_Orig["Code"].isin(Code_List)].reset_index(drop=True)
    return (Filtered_DF, ET_DF)

In [34]:
# Filter for storms with ET Begin SLP <= 1005 and ET Complete SLP <= 1005
Control_Data_V7, Control_ET_V7 = SLP_Filter(Control_Data_V6, Control_ET_V6)
RCP45_Data_V7, RCP45_ET_V7 = SLP_Filter(RCP45_Data_V6, RCP45_ET_V6)
RCP85_Data_V7, RCP85_ET_V7 = SLP_Filter(RCP85_Data_V6, RCP85_ET_V6)

In [35]:
# Filter for storms based on ET location
def ET_Loc_Filter(Data_DF_Orig, ET_DF_Orig):
# Filter out ETs near European coast
    ET_DF_Filt = ET_DF_Orig[(ET_DF_Orig["ET Complete Lon"] < -10)]
# Filter out ETs deep inland within North America
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Begin Lat"] < 32) | (ET_DF_Filt["ET Begin Lon"] > -83)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Begin Lat"] < 34) | (ET_DF_Filt["ET Begin Lon"] > -81)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Begin Lat"] < 36) | (ET_DF_Filt["ET Begin Lon"] > -78)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Begin Lat"] < 40) | (ET_DF_Filt["ET Begin Lon"] > -76)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Begin Lat"] < 42) | (ET_DF_Filt["ET Begin Lon"] > -74)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Begin Lat"] < 45) | (ET_DF_Filt["ET Begin Lon"] > -71)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Begin Lat"] < 48) | (ET_DF_Filt["ET Begin Lon"] > -67)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Begin Lat"] < 52) | (ET_DF_Filt["ET Begin Lon"] > -62)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Complete Lat"] < 31) | (ET_DF_Filt["ET Complete Lon"] > -83)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Complete Lat"] < 34) | (ET_DF_Filt["ET Complete Lon"] > -81)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Complete Lat"] < 36) | (ET_DF_Filt["ET Complete Lon"] > -78)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Complete Lat"] < 40) | (ET_DF_Filt["ET Complete Lon"] > -76)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Complete Lat"] < 45) | (ET_DF_Filt["ET Complete Lon"] > -72)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Complete Lat"] < 49) | (ET_DF_Filt["ET Complete Lon"] > -68)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Complete Lat"] < 52) | (ET_DF_Filt["ET Complete Lon"] > -62)]
    ET_DF = ET_DF_Filt.reset_index(drop=True)
    Code_List = ET_DF["Code"]
    Filtered_DF = Data_DF_Orig[Data_DF_Orig["Code"].isin(Code_List)].reset_index(drop=True)
    return (Filtered_DF, ET_DF)

In [36]:
# Filter for storms based on ET location
Control_Data_V8, Control_ET_V8 = ET_Loc_Filter(Control_Data_V7, Control_ET_V7)
RCP45_Data_V8, RCP45_ET_V8 = ET_Loc_Filter(RCP45_Data_V7, RCP45_ET_V7)
RCP85_Data_V8, RCP85_ET_V8 = ET_Loc_Filter(RCP85_Data_V7, RCP85_ET_V7)

In [37]:
# Filter out storms with missing composite data
def Missing_Filter(Data_DF_Orig, ET_DF_Orig, Missing_Codes):
    ET_DF = ET_DF_Orig[~ET_DF_Orig["Code"].isin(Missing_Codes)].reset_index(drop=True)
    Code_List = ET_DF["Code"]
    Filtered_DF = Data_DF_Orig[Data_DF_Orig["Code"].isin(Code_List)].reset_index(drop=True)
    return (Filtered_DF, ET_DF)

Missing_Codes = ['TC190302', 'TC195509', 'TC197112', 'TC197802', 'TC198107',
'TC200201', 'TC200712', 'TC200904', 'TC201509', 'TC201604', 'TC202708', 'TC203001', 'TC203007', \
'TC203401', 'TC203701', 'TC203907', 'TC204403', 'TC205305', 'TC205306', 'TC206106', \
'TC206601', 'TC207505', 'TC207602', 'TC207607', 'TC208205', 'TC208304', 'TC209310', \
'TC210203', 'TC211108', 'TC211407', 'TC211805', 'TC211906', 'TC212401', 'TC212703', 'TC212804', \
'TC213505', 'TC213606', 'TC214004', 'TC214005', 'TC214309', 'TC214405', 'TC214901', 'TC215104', 'TC215306', 'TC215904', \
'TC216604', 'TC217104', 'TC217402', 'TC217802', 'TC218206', 'TC218508', 'TC219204', 'TC219401']

In [38]:
# Filter out storms with missing composite data
Control_Data_V9, Control_ET_V9 = Missing_Filter(Control_Data_V8, Control_ET_V8, Missing_Codes)
RCP45_Data_V9, RCP45_ET_V9 = Missing_Filter(RCP45_Data_V8, RCP45_ET_V8, Missing_Codes)
RCP85_Data_V9, RCP85_ET_V9 = Missing_Filter(RCP85_Data_V8, RCP85_ET_V8, Missing_Codes)

In [39]:
# Count transition type
def Trans_Type_Percent(Series):
    Percents = Series.value_counts(normalize=True).reindex([0, 1, 2], fill_value=0)
    return (Percents)

In [40]:
# Calculate percentages of ET completion
ET_Percents = pandas.DataFrame({"Control": Trans_Type_Percent(Control_ET_V4["Trans Type"]), \
                                "RCP4.5": Trans_Type_Percent(RCP45_ET_V4["Trans Type"]), \
                                "RCP8.5": Trans_Type_Percent(RCP85_ET_V4["Trans Type"])})
ET_Percents.index = ["No ET", "ET Incomplete", "ET Complete"]
ET_Percents

,Control,RCP4.5,RCP8.5
No ET,0.270690,0.251816,0.233333
ET Incomplete,0.220690,0.220339,0.188889
ET Complete,0.508621,0.527845,0.577778


In [41]:
# Create table for number of storms within each subset
Control_Count = [len(Control_ET_V2), len(Control_ET_V5), len(Control_ET_V6), len(Control_ET_V9)]
RCP45_Count = [len(RCP45_ET_V2), len(RCP45_ET_V5), len(RCP45_ET_V6), len(RCP45_ET_V9)]
RCP85_Count = [len(RCP85_ET_V2), len(RCP85_ET_V5), len(RCP85_ET_V6), len(RCP85_ET_V9)]
Storm_Counts = pandas.DataFrame({"Control": Control_Count, "RCP4.5": RCP45_Count, "RCP8.5": RCP85_Count})
Storm_Counts.index = ["Raw", "Dataset A", "Subset B", "Subset C"]
Storm_Counts

,Control,RCP4.5,RCP8.5
Raw,918,746,650
Dataset A,580,413,360
Subset B,295,218,208
Subset C,218,145,138


In [42]:
# Output DF to csv file
def Output_File(DF, Scenario, Type):
    File_Name = str(Scenario+'_'+Type+'_Output.csv')
    DF.to_csv(Output_Diri+File_Name)

In [43]:
# Output control files
Output_File(Control_Data_V5, "Control", "Data_DatasetA")
Output_File(Control_Data_V6, "Control", "Data_SubsetB")
Output_File(Control_Data_V9, "Control", "Data_SubsetC")
Output_File(Control_ET_V5, "Control", "ET_DatasetA")
Output_File(Control_ET_V6, "Control", "ET_SubsetB")
Output_File(Control_ET_V9, "Control", "ET_SubsetC")
Output_File(Control_Codes_V5, "Control", "Codes")

In [44]:
# Output RCP4.5 files
Output_File(RCP45_Data_V5, "RCP45", "Data_DatasetA")
Output_File(RCP45_Data_V6, "RCP45", "Data_SubsetB")
Output_File(RCP45_Data_V9, "RCP45", "Data_SubsetC")
Output_File(RCP45_ET_V5, "RCP45", "ET_DatasetA")
Output_File(RCP45_ET_V6, "RCP45", "ET_SubsetB")
Output_File(RCP45_ET_V9, "RCP45", "ET_SubsetC")
Output_File(RCP45_Codes_V5, "RCP45", "Codes")

In [45]:
# Output RCP8.5 files
Output_File(RCP85_Data_V5, "RCP85", "Data_DatasetA")
Output_File(RCP85_Data_V6, "RCP85", "Data_SubsetB")
Output_File(RCP85_Data_V9, "RCP85", "Data_SubsetC")
Output_File(RCP85_ET_V5, "RCP85", "ET_DatasetA")
Output_File(RCP85_ET_V6, "RCP85", "ET_SubsetB")
Output_File(RCP85_ET_V9, "RCP85", "ET_SubsetC")
Output_File(RCP85_Codes_V5, "RCP85", "Codes")

In [46]:
# Output storm counts table
Storm_Counts.to_csv("Figures/Table_1.csv")